# **Customer-Level Churn Dataset Preparation and Feature Engineering – Olist E-Commerce**

# Workflow (Step-by-Step Code Process):

Load Raw Datasets → Merge Customers, Orders, Items, Payments, Reviews → Convert Datetime Columns → Aggregate to Customer-Level → Derive Features (Recency, Avg Items per Order, etc.) → Fill Missing Values → Apply Business Rule for Churn → Export Clean Customer-Level Dataset

# Imports and Warnings

In [1]:
import pandas as pd
import numpy as np

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Load Datasets

In [2]:
customer = pd.read_csv('olist_customers_dataset.csv')
item = pd.read_csv('olist_order_items_dataset.csv')
payment = pd.read_csv('olist_order_payments_dataset.csv')
review = pd.read_csv('olist_order_reviews_dataset.csv')
order = pd.read_csv('olist_orders_dataset.csv')


# Merge Datasets

In [ ]:
# Merge customers+orders 
cust_orders = pd.merge(order, customer, on='customer_id', how='left')

# Add order items
cust_orders_items = pd.merge(cust_orders, item, on='order_id', how='left')

# Add payment information
cust_orders_items_pay = pd.merge(cust_orders_items, payment, on='order_id', how='left')

# Add order review
final_order_level = pd.merge(cust_orders_items_pay, review, on='order_id', how='left')

# final_order_level now contains all order-level info: customer, items, payments, reviews
print(final_order_level.shape)


(119143, 28)


In [4]:
pd.options.display.max_columns = None
final_order_level.head()


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,payment_sequential,payment_type,payment_installments,payment_value,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,1.0,credit_card,1.0,18.12,a54f0611adc9ed256b57ede6b6eb5114,4.0,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,3.0,voucher,1.0,2.00,a54f0611adc9ed256b57ede6b6eb5114,4.0,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,2.0,voucher,1.0,18.59,a54f0611adc9ed256b57ede6b6eb5114,4.0,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1.0,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,1.0,boleto,1.0,141.46,8d5266042046a06655c8db133d120ba5,4.0,Muito boa a loja,Muito bom o produto.,2018-08-08 00:00:00,2018-08-08 18:37:50
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1.0,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,1.0,credit_card,3.0,179.12,e73b67b67587f7644d5bd1a52deb1b01,5.0,NaN,NaN,2018-08-18 00:00:00,2018-08-22 19:07:58


# Datetime Conversion & Reference Date

In [5]:
# Convert order date to datetime format
final_order_level['order_purchase_timestamp'] = pd.to_datetime(final_order_level['order_purchase_timestamp'])

# Set reference date as latest order date
reference_date = final_order_level['order_purchase_timestamp'].max()

# Aggregate to Customer Level

--- *Customer-Level Aggregation (RFM + Extended Features)* ---

* Aggregating order-level data to customer-level for churn analysis.
* RFM mapping:
* Recency (R) → days_since_last_purchase: days since last purchase, higher means higher churn risk
* Frequency (F) → total_orders: number of orders, measures engagement
* Monetary (M) → total_spent / avg_order_value: total and average spend, measures customer value
* Extra features for richer customer insights:
* avg_items_per_order → purchase pattern
* preferred_payment_type → behavioral preference
* avg_review_score → satisfaction
* avg_product_price → typical spending per product

In [6]:
customer_level = final_order_level.groupby('customer_unique_id').agg(
    total_orders = ('order_id', 'nunique'),                          # number of orders (Frequency)
    last_purchase = ('order_purchase_timestamp', 'max'),             # last order date (Recency helper)
    total_spent = ('payment_value', 'sum'),                          # total spend (Monetary)
    avg_order_value = ('payment_value', 'mean'),                     # avg order value
    preferred_payment_type = ('payment_type', lambda x: x.mode()[0] if not x.mode().empty else np.nan),
    total_items = ('order_item_id', 'count'),                        # total items purchased (helper for avg_items_per_order)
    avg_product_price = ('price', 'mean'),                           # avg price per product
    avg_review_score = ('review_score', 'mean')                      # average review score (satisfaction)
).reset_index()

# Derived Features

In [7]:
# Recency → days since last purchase
customer_level['days_since_last_purchase'] = (reference_date - customer_level['last_purchase']).dt.days

# Avg items per order → purchase behavior
customer_level['avg_items_per_order'] = customer_level['total_items'] / customer_level['total_orders']

# Drop helper columns used for calculations
customer_level.drop(columns=['last_purchase', 'total_items'], inplace=True)

# Check shape and preview
print(customer_level.shape)
customer_level.head()

(96096, 9)


,customer_unique_id,total_orders,total_spent,avg_order_value,preferred_payment_type,avg_product_price,avg_review_score,days_since_last_purchase,avg_items_per_order
0,0000366f3b9a7992bf8c76cfdf3221e2,1,141.90,141.90,credit_card,129.90,5.0,160,1.0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,27.19,27.19,credit_card,18.90,4.0,163,1.0
2,0000f46a3911fa3c0805444483337064,1,86.22,86.22,credit_card,69.00,3.0,585,1.0
3,0000f6ccb0745a6a4b88665a16c9f078,1,43.62,43.62,credit_card,25.99,4.0,369,1.0
4,0004aac84e0df4da2b147fca70cf8255,1,196.89,196.89,credit_card,180.00,5.0,336,1.0


In [8]:
customer_level.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 96096 entries, 0 to 96095
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   customer_unique_id        96096 non-null  object 
 1   total_orders              96096 non-null  int64  
 2   total_spent               96096 non-null  float64
 3   avg_order_value           96095 non-null  float64
 4   preferred_payment_type    96095 non-null  object 
 5   avg_product_price         95420 non-null  float64
 6   avg_review_score          95380 non-null  float64
 7   days_since_last_purchase  96096 non-null  int64  
 8   avg_items_per_order       96096 non-null  float64
dtypes: float64(5), int64(2), object(2)
memory usage: 6.6+ MB


# Handle Missing Values

In [9]:
# avg_order_value: fill with median
median_order_value = customer_level['avg_order_value'].median()
customer_level['avg_order_value'].fillna(median_order_value, inplace=True)

# preferred_payment_type: fill with mode
mode_payment_type = customer_level['preferred_payment_type'].mode()[0]
customer_level['preferred_payment_type'].fillna(mode_payment_type, inplace=True)

# avg_product_price: fill with median
median_product_price = customer_level['avg_product_price'].median()
customer_level['avg_product_price'].fillna(median_product_price, inplace=True)

# avg_review_score: fill with neutral score (3.0)
customer_level['avg_review_score'].fillna(3.0, inplace=True)

# Check nulls again
print(customer_level.isnull().sum())


customer_unique_id          0
total_orders                0
total_spent                 0
avg_order_value             0
preferred_payment_type      0
avg_product_price           0
avg_review_score            0
days_since_last_purchase    0
avg_items_per_order         0
dtype: int64


# Define Churn

In [ ]:
# Business rule:
# Customer is churned if days_since_last_purchase > 240 days
threshold_days = 240

# 1 = churned, 0 = active
customer_level['churn'] = customer_level['days_since_last_purchase'].apply(lambda x: 1 if x > threshold_days else 0)


# Export Dataset

In [11]:
# Export the cleaned customer-level dataset to CSV
customer_level.to_csv("customer_churn_dataset.csv", index=False)

print("File successfully saved as customer_churn_dataset.csv")


File successfully saved as customer_churn_dataset.csv
